# Raw Face Training in Colab

This notebook trains on exported face images from Django, then reports whether the model reaches at least 60% validation accuracy.

It supports both Google Drive bundles and Firebase Firestore sources.


In [ ]:
!pip -q install pandas numpy scikit-learn pillow matplotlib seaborn firebase-admin joblib

In [ ]:
import base64
import json
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance, ImageOps
from joblib import dump

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)


In [ ]:
try:
    from google.colab import drive, userdata  # type: ignore
except Exception:
    drive = None
    userdata = None


def secret(name, default=''):
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return default

if drive is not None:
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        print(f'Drive mount skipped: {exc}')

SOURCE_MODE = secret('COLAB_SOURCE_MODE', 'drive').strip().lower()
DRIVE_EXPORT_DIR = Path(secret('COLAB_DRIVE_EXPORT_DIR', '/content/drive/MyDrive/colab_exports'))
DRIVE_OUTPUT_DIR = Path(secret('COLAB_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/colab_output'))
DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

firebase_db = None
if SOURCE_MODE == 'firebase':
    import firebase_admin
    from firebase_admin import credentials, firestore

    firebase_creds = secret('FIREBASE_CREDENTIALS_PATH', '')
    if not firebase_creds:
        raise ValueError('Set FIREBASE_CREDENTIALS_PATH to your Firebase service account JSON path.')
    if not firebase_admin._apps:
        firebase_admin.initialize_app(credentials.Certificate(firebase_creds))
    firebase_db = firestore.client()


def latest_zip_in(directory):
    directory = Path(directory)
    if not directory.exists():
        return None
    zips = sorted(directory.glob('*.zip'), key=lambda item: item.stat().st_mtime, reverse=True)
    return zips[0] if zips else None

latest_bundle = latest_zip_in(DRIVE_EXPORT_DIR)
BUNDLE_PATH = secret('COLAB_BUNDLE_PATH', str(latest_bundle) if latest_bundle else str(DRIVE_EXPORT_DIR / 'colab_bundle.zip'))
OUTPUT_DIR = Path(secret('COLAB_OUTPUT_DIR', str(DRIVE_OUTPUT_DIR)))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Source mode:', SOURCE_MODE)
print('Bundle path:', BUNDLE_PATH)
print('Output dir:', OUTPUT_DIR)
if latest_bundle:
    print('Latest Drive bundle:', latest_bundle)


In [ ]:
def load_bundle(bundle_path):
    bundle_path = Path(bundle_path)
    if bundle_path.is_dir():
        return bundle_path
    if not bundle_path.exists():
        raise FileNotFoundError(f'Bundle not found: {bundle_path}')
    extract_dir = OUTPUT_DIR / bundle_path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)
    if bundle_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(bundle_path, 'r') as archive:
            archive.extractall(extract_dir)
        return extract_dir
    return bundle_path.parent


def firestore_collection_df(collection_name):
    docs = firebase_db.collection(collection_name).stream()
    records = []
    for doc in docs:
        payload = doc.to_dict() or {}
        payload['_doc_id'] = doc.id
        records.append(payload)
    return pd.DataFrame(records)


def load_data_frame(source, drive_path=None, firestore_collection=None):
    if source == 'firebase':
        if not firestore_collection:
            return pd.DataFrame()
        return firestore_collection_df(firestore_collection)
    if drive_path is None:
        return pd.DataFrame()
    return pd.read_csv(drive_path)


In [ ]:
if SOURCE_MODE == 'firebase':
    face_manifest = pd.DataFrame()
    facial_records = load_data_frame('firebase', firestore_collection='facial_biometrics')
    clients_df = load_data_frame('firebase', firestore_collection='clients')
    payments_df = load_data_frame('firebase', firestore_collection='payments')
    overdue_df = load_data_frame('firebase', firestore_collection='overdue_snapshot')
else:
    bundle_dir = load_bundle(BUNDLE_PATH)
    print('Loaded bundle dir:', bundle_dir)
    print('Files:', sorted([p.name for p in bundle_dir.iterdir()]))
    face_manifest = load_data_frame('drive', bundle_dir / 'face_images_manifest.csv')
    facial_path = bundle_dir / 'facial_biometrics.jsonl'
    facial_records = pd.read_json(facial_path, lines=True) if facial_path.exists() else pd.DataFrame()
    clients_df = load_data_frame('drive', bundle_dir / 'clients.csv')
    payments_df = load_data_frame('drive', bundle_dir / 'payments.csv')
    overdue_df = load_data_frame('drive', bundle_dir / 'overdue_snapshot.csv')


## Load Face Images

For Drive bundles, the notebook uses the exported image files directly. For Firebase, it expects `sample_image_b64` on the `facial_biometrics` documents.


In [ ]:
def load_image_from_path(path, size=(96, 96)):
    image = Image.open(path).convert('L')
    image = ImageOps.fit(image, size, method=Image.Resampling.LANCZOS)
    return np.asarray(image, dtype=np.float32) / 255.0


def load_image_from_b64(value, size=(96, 96)):
    raw = base64.b64decode(value)
    from io import BytesIO
    img = Image.open(BytesIO(raw)).convert('L')
    img = ImageOps.fit(img, size, method=Image.Resampling.LANCZOS)
    return np.asarray(img, dtype=np.float32) / 255.0


def augment_image(image_array):
    image = Image.fromarray((image_array * 255).astype(np.uint8), mode='L')
    variants = [image]
    variants.append(ImageOps.mirror(image))
    variants.append(ImageEnhance.Brightness(image).enhance(1.12))
    variants.append(ImageEnhance.Contrast(image).enhance(1.10))
    return [np.asarray(ImageOps.fit(v, (96, 96), method=Image.Resampling.LANCZOS), dtype=np.float32) / 255.0 for v in variants]

face_rows = []
face_labels = []
face_meta = []

if SOURCE_MODE == 'firebase':
    if facial_records.empty:
        print('No facial biometrics found in Firebase.')
    else:
        for _, row in facial_records.iterrows():
            sample_b64 = row.get('sample_image_b64', '')
            if not isinstance(sample_b64, str) or not sample_b64:
                continue
            base = load_image_from_b64(sample_b64)
            for variant_idx, variant in enumerate(augment_image(base)):
                face_rows.append(variant.reshape(-1))
                face_labels.append(row.get('username') or str(row.get('user_id', 'unknown')))
                face_meta.append({
                    'username': row.get('username'),
                    'variant_idx': variant_idx,
                    'quality_score': row.get('quality_score', None),
                    'confidence': row.get('confidence', None),
                    'source': 'firebase',
                })
else:
    if face_manifest.empty:
        print('No face image manifest found. Export with --include-models.')
    else:
        for _, row in face_manifest.iterrows():
            image_path = bundle_dir / row['sample_image_file']
            if not image_path.exists():
                continue
            base = load_image_from_path(image_path)
            for variant_idx, variant in enumerate(augment_image(base)):
                face_rows.append(variant.reshape(-1))
                face_labels.append(row['username'] if isinstance(row['username'], str) and row['username'] else str(row['user_id']))
                face_meta.append({
                    'username': row['username'],
                    'variant_idx': variant_idx,
                    'quality_score': row.get('quality_score', None),
                    'confidence': row.get('confidence', None),
                    'source': 'drive',
                })

face_X = np.vstack(face_rows) if face_rows else np.empty((0, 96 * 96))
face_y = np.array(face_labels) if face_labels else np.array([])
face_meta_df = pd.DataFrame(face_meta) if face_meta else pd.DataFrame()
print('Samples:', len(face_y), 'Unique labels:', len(set(face_labels)))
display(face_meta_df.head())


## Train Face Classifier

This is a baseline classifier, not a production face-recognition stack. The goal is to see whether the exported images are good enough to clear 60% validation accuracy.


In [ ]:
if len(face_y) < 8 or len(set(face_y)) < 2:
    print('Not enough face image data to train a reliable classifier yet.')
else:
    X_train, X_test, y_train, y_test = train_test_split(
        face_X, face_y, test_size=0.25, random_state=42, stratify=face_y
    )
    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced_subsample',
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f'Validation accuracy: {acc:.2%}')
    print(classification_report(y_test, preds, zero_division=0))
    print(confusion_matrix(y_test, preds))
    if acc >= 0.60:
        print('Target met: validation accuracy is 60% or higher.')
    else:
        print('Target not met yet. Add more raw images per person or improve capture consistency.')
    dump(model, OUTPUT_DIR / 'face_classifier.joblib')
    print('Saved model to', OUTPUT_DIR / 'face_classifier.joblib')


## Overdue and Payment Analysis

This view works with both Drive and Firebase sources.


In [ ]:
clients = clients_df.copy()
if not clients.empty:
    if 'balance' in clients.columns:
        clients['balance'] = pd.to_numeric(clients['balance'], errors='coerce').fillna(0)
    else:
        clients['balance'] = 0
    clients['days_overdue'] = pd.to_numeric(clients.get('days_overdue', 0), errors='coerce').fillna(0)
    clients['is_overdue'] = clients['status'].isin(['Overdue', 'Disconnected']).astype(int)

    display(clients[['client_id', 'name', 'status', 'balance', 'days_overdue', 'plan']].sort_values('days_overdue', ascending=False).head(20))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.countplot(data=clients, x='status', ax=axes[0], palette='Set2')
    axes[0].set_title('Client status')
    sns.countplot(data=payments_df, x='status', ax=axes[1], palette='Set3')
    axes[1].set_title('Payment status')
    plt.tight_layout()
else:
    print('No client data available in this source.')


## Next Step

If you use Firebase for face training, make sure you sync the `facial_biometrics` collection with `sample_image_b64` values first. If you use Drive, the bundle export already includes the raw image files.
